# Results 5 — Gene-level pleiotropy

The gene pleiotropy score (gPS), its model, its association with 21 gene sets, and how both
pleiotropy scores grew over time.

| file | panel |
| --- | --- |
| `Fig4A_stats_gene_pleiotropy.csv`, `Fig4A_stats_variant_pleiotropy.csv`, `Fig4A_stats_gene_coverage.csv` | Figure 4a |
| `gene_pleiotropy_full_model.csv` | Figure 4b |
| `gene_pleiotropy_by_category.csv` | Figure 4c |

The pathway enrichment behind Supplementary Table 3 needs the Enrichr gene-set files, which are
not available; see GAPS.md.

In [ ]:
import math

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

from manuscript_methods import clusters, paper

numbers = {}
gene_table = pd.read_parquet(paper.derived("gene_table"))
print("genes:", len(gene_table))

## The score

In [ ]:
numbers["R5.01"] = int((gene_table["uniqueDiseases"] > 1).sum())
numbers["R5.02"] = round(float(gene_table["uniqueDiseases"].mean()), 2)
numbers["R5.03"] = int(gene_table["uniqueDiseases"].max())
numbers["R5.04"] = int((gene_table["uniqueTherapeuticAreas"] > 1).sum())
numbers["R5.05"] = round(float(gene_table["uniqueTherapeuticAreas"].mean()), 2)
numbers["R5.06"] = int(gene_table["uniqueTherapeuticAreas"].max())
numbers["R5.07"] = round(float(spearmanr(gene_table["uniqueDiseases"], gene_table["uniqueTherapeuticAreas"])[0]), 2)
numbers["R5.13"] = round(float(spearmanr(gene_table["uniqueVariants"], gene_table["uniqueDiseases"])[0]), 2)

by_symbol = gene_table.set_index("approvedSymbol")
for key, symbol in [("R5.08", "FTO"), ("R5.09", "APOE"), ("R5.10", "ABO"), ("R5.11", "CDKN2B")]:
    numbers[key] = int(by_symbol.loc[symbol, "uniqueDiseases"])
numbers["R5.12"] = int(by_symbol.loc["CDKN2B", "uniqueTherapeuticAreas"])

constraint = gene_table[["lofConstraint", "misConstraint"]].dropna()
numbers["R5.22"] = round(float(pearsonr(constraint["lofConstraint"], constraint["misConstraint"])[0]), 3)

print({k: numbers[k] for k in sorted(numbers)})
print()
print(
    gene_table.nlargest(6, "uniqueDiseases")[["approvedSymbol", "uniqueDiseases", "uniqueTherapeuticAreas"]].to_string(
        index=False
    )
)

## Figure 4b — negative binomial model of gPS

Nine min-max scaled covariates. `tissueSpecificityBinary` marks genes whose transcriptional
profile is specific to few tissues (specificity above 0.75); it needs
`target_prioritisation`, and the model falls back to eight covariates when that is absent.

In [ ]:
model_frame = gene_table.copy()
for column in ["lofConstraint", "misConstraint", "tissueSpecificity"]:
    model_frame[column] = model_frame[column].fillna(model_frame[column].mean())

has_tissue = model_frame["tissueSpecificity"].notna().any()
model_frame["tissueSpecificityBinary"] = (model_frame["tissueSpecificity"] > 0.75).astype(int)

covariates = [
    "maxEQTLColoc",
    "maxPQTLColoc",
    "maxVEP",
    "maxEffectiveSampleSize",
    "lofConstraint",
    "misConstraint",
    "geneLength",
    "pathwayCount",
]
if has_tissue:
    covariates.append("tissueSpecificityBinary")
else:
    print("tissueSpecificity unavailable: fitting eight covariates, R5.14 and R5.21 are blocked")

for column in covariates:
    span = model_frame[column].max() - model_frame[column].min()
    model_frame[f"{column}Normalised"] = 0.0 if span == 0 else (model_frame[column] - model_frame[column].min()) / span
normalised = [f"{c}Normalised" for c in covariates]

In [ ]:
def fit(columns):
    """Negative binomial fit of gPS on the given covariates."""
    x = sm.add_constant(model_frame[columns].copy())
    return sm.NegativeBinomial(model_frame["uniqueDiseases"], x).fit(maxiter=1000, disp=False), x


records = []
for model_type, column_sets in [("Univariate", [[c] for c in normalised]), ("Multi", [normalised])]:
    for columns in column_sets:
        model, x = fit(columns)
        ci = model.conf_int()
        for covariate in columns:
            records.append(
                {
                    "covariate": covariate,
                    "model_type": model_type,
                    "coefficient": model.params[covariate],
                    "std_error": model.bse[covariate],
                    "p_value": model.pvalues[covariate],
                    "ci_lower": ci.loc[covariate, 0],
                    "ci_upper": ci.loc[covariate, 1],
                }
            )

coefficients = pd.DataFrame(records)
coefficients["y_numerical"] = coefficients["covariate"].map({c: i for i, c in enumerate(normalised)})
coefficients["y_plot"] = coefficients["y_numerical"] + np.where(coefficients["model_type"] == "Univariate", -0.1, 0.1)
coefficients.to_csv(paper.derived("gene_pleiotropy_full_model.csv"), index=False)

joint, x_joint = fit(normalised)
numbers["R5.14"] = round(float(np.corrcoef(model_frame["uniqueDiseases"], joint.predict(x_joint))[0, 1] ** 2), 2)

univariate = coefficients[coefficients["model_type"] == "Univariate"].set_index("covariate")["coefficient"]
mapping = {
    "R5.15": "maxEffectiveSampleSizeNormalised",
    "R5.16": "lofConstraintNormalised",
    "R5.17": "misConstraintNormalised",
    "R5.18": "pathwayCountNormalised",
    "R5.19": "geneLengthNormalised",
    "R5.20": "maxVEPNormalised",
    "R5.21": "tissueSpecificityBinaryNormalised",
}
for key, covariate in mapping.items():
    if covariate in univariate.index:
        numbers[key] = round(float(univariate[covariate]), 2)
print({k: numbers[k] for k in ["R5.14", *mapping] if k in numbers})
coefficients.round(4)

## Figure 4c — gPS against membership in 21 gene sets

One logistic regression per gene set: membership on log2(gPS), over the genes that have a gPS
and belong to at least one of the included sets. The odds ratio is per doubling of gPS.

In [ ]:
CATEGORY_LABELS = {
    "all_diseases": "GWAS",
    "cancer_ChEMBL": "Cancer ChEMBL",
    "gene_burden": "Gene-based analysis",
    "omim": "OMIM",
    "cancer_driver_gene": "Cancer Driver (COSMIC)",
    "withdrawn_drug": "Withdrawn Drug",
    "non_cancer_ChEMBL": "Non-Cancer ChEMBL",
    "essential_gene": "Essential Gene (DepMap)",
    "gwas_eQTL": "GWAS with eQTL evidence",
    "gwas_with_pav": "GWAS with PAV evidence",
    "liable_target": "Known safety events",
    "orphanet": "Orphanet",
    "ChEMBL": "ChEMBL",
    "dd_related": "DD panel (gene2phenotype)",
    "pharmacogene": "Pharmacogenetics - Toxicity",
    "trial_safety_concern": "Trial Safety",
    "mouse_ko_mortality": "Mouse KO Mortality",
    "All genes": "All protein-coding genes",
    "fusil_CL": "Cellular lethal (FUSIL)",
    "fusil_DL": "Developmental lethal (FUSIL)",
    "fusil_SV": "Subviable (FUSIL)",
    "fusil_VP": "Viable with phenotype (FUSIL)",
    "fusil_VN": "Viable with no phenotype (FUSIL)",
    "lof_constr_Q4": "Q4 LoF constraint",
    "lof_constr_Q3": "Q3 LoF constraint",
    "lof_constr_Q2": "Q2 LoF constraint",
    "lof_constr_Q1": "Q1 LoF constraint",
    "non_essential": "Non-essential Gene (DepMap)",
    "distant_ortholog": "Drosophila distant orthologs",
    "human_ko": "Human Knockout",
}
INCLUDED = [
    "Gene-based analysis",
    "OMIM",
    "Cancer Driver (COSMIC)",
    "Withdrawn Drug",
    "Essential Gene (DepMap)",
    "Known safety events",
    "Orphanet",
    "ChEMBL",
    "DD panel (gene2phenotype)",
    "Trial Safety",
    "Mouse KO Mortality",
    "Cellular lethal (FUSIL)",
    "Developmental lethal (FUSIL)",
    "Subviable (FUSIL)",
    "Viable with phenotype (FUSIL)",
    "Viable with no phenotype (FUSIL)",
    "Q4 LoF constraint",
    "Q1 LoF constraint",
    "Non-essential Gene (DepMap)",
    "Drosophila distant orthologs",
    "Human Knockout",
]

sets = pd.read_parquet(paper.derived("gene_sets"))
sets["geneSet"] = sets["geneSet"].replace(CATEGORY_LABELS)
sets = sets[sets["geneSet"].isin(INCLUDED)].drop_duplicates(["geneId", "geneSet"])
set_totals = sets.groupby("geneSet")["geneId"].nunique().to_dict()
print("gene sets:", len(set_totals))

In [ ]:
gps = gene_table[["geneId", "uniqueDiseases"]].copy()
in_any_set = gps[gps["geneId"].isin(set(sets["geneId"]))].copy()
in_any_set["log2_gps"] = np.log2(in_any_set["uniqueDiseases"])
print("genes with a gPS in at least one included set:", len(in_any_set))

records = []
for gene_set in sorted(set_totals):
    members = set(sets.loc[sets["geneSet"] == gene_set, "geneId"])
    y = in_any_set["geneId"].isin(members).astype(int)
    x = sm.add_constant(in_any_set["log2_gps"].astype(float))
    model = sm.Logit(y, x).fit(disp=0)
    log_or = model.params["log2_gps"]
    ci = model.conf_int().loc["log2_gps"]
    total = set_totals[gene_set]
    overlap = 100 * int(y.sum()) / total
    records.append(
        {
            "category": gene_set,
            "label": f"{gene_set} ({total}/{overlap:.1f}%)",
            "odds_ratio": float(np.exp(log_or)),
            "log_odds_ratio": float(log_or),
            "ci_lower": float(np.exp(ci[0])),
            "ci_upper": float(np.exp(ci[1])),
            "log_ci_lower": float(ci[0]),
            "log_ci_upper": float(ci[1]),
            "p_value": float(model.pvalues["log2_gps"]),
            "n_in_category": int(y.sum()),
            "total_in_category": total,
            "pct_overlap": overlap,
        }
    )

categories = pd.DataFrame(records).sort_values("log_odds_ratio")
categories["fdr"] = multipletests(categories["p_value"], method="fdr_bh")[1]
categories.drop(columns=["n_in_category", "total_in_category", "pct_overlap"]).to_csv(
    paper.derived("gene_pleiotropy_by_category.csv"), index=False
)
categories[["category", "odds_ratio", "p_value", "fdr", "total_in_category"]].round(4).to_string(index=False)

In [ ]:
by_category = categories.set_index("category")
numbers["R5.26"] = len(categories)
numbers["R5.27"] = int(((by_category["fdr"] < 0.05) & (by_category["odds_ratio"] > 1)).sum())
for key, category in [
    ("R5.28", "Cancer Driver (COSMIC)"),
    ("R5.29", "Mouse KO Mortality"),
    ("R5.30", "DD panel (gene2phenotype)"),
    ("R5.31", "Trial Safety"),
    ("R5.32", "Withdrawn Drug"),
    ("R5.34", "Known safety events"),
    ("R5.36", "Q4 LoF constraint"),
    ("R5.37", "Q1 LoF constraint"),
    ("R5.38", "Human Knockout"),
]:
    numbers[key] = round(float(by_category.loc[category, "odds_ratio"]), 2)
numbers["R5.33"] = int(by_category.loc["Withdrawn Drug", "total_in_category"])
numbers["R5.35"] = int(by_category.loc["Known safety events", "total_in_category"])
print({k: numbers[k] for k in sorted(numbers) if k >= "R5.26"})

## Figure 4a — both scores over time

For each year, the credible sets published up to that year are re-clustered and the mean
per-cluster disease count taken (variant pleiotropy); the same cumulative subset gives the mean
diseases per gene (gene pleiotropy) and the mean lead variants per gene (coverage).

In [ ]:
YEARS = range(2006, 2026)

credible_sets = clusters.load_credible_sets()
study_years = pd.read_parquet(paper.derived("study_annotation"))[["studyId", "year"]]
credible_sets = credible_sets.merge(study_years, on="studyId", how="left")
credible_sets["year"] = credible_sets["year"].fillna(2024).astype(int)
all_edges = clusters.load_edges(set(credible_sets["studyLocusId"]))
print("credible sets:", len(credible_sets), "edges:", len(all_edges))

records = []
for year in YEARS:
    subset = credible_sets[credible_sets["year"] <= year]
    if subset.empty:
        continue
    locus_ids = set(subset["studyLocusId"])
    edges = [(a, b) for a, b in all_edges if a in locus_ids and b in locus_ids]
    components = clusters.cluster(list(zip(subset["studyLocusId"], subset["variantId"])), edges)
    counts = clusters.cluster_table(subset, components)["uniqueDiseases"].to_numpy()
    n = len(counts)
    sd = float(np.std(counts, ddof=1)) if n > 1 else 0.0
    records.append(
        {
            "year": year,
            "mean": float(counts.mean()),
            "sd": sd,
            "se": sd / math.sqrt(n) if n else None,
            "n_agg": n,
            "n_count": n,
        }
    )

variant_pleiotropy = pd.DataFrame(records)
variant_pleiotropy.to_csv(paper.derived("Fig4A_stats_variant_pleiotropy.csv"))
variant_pleiotropy.tail(4)

In [ ]:
genes = pd.read_parquet(paper.derived("prioritised_genes_diseases"))[["geneId", "variantId", "diseaseIds", "year"]]

gene_rows, coverage_rows = [], []
for year in YEARS:
    subset = genes[genes["year"] <= year]
    if subset.empty:
        continue
    per_gene = subset.groupby("geneId").agg(
        diseases=("diseaseIds", lambda col: len({d for arr in col if arr is not None for d in arr})),
        variants=("variantId", "nunique"),
    )
    for rows, column in [(gene_rows, "diseases"), (coverage_rows, "variants")]:
        values = per_gene[column]
        rows.append(
            {"year": year, "mean": float(values.mean()), "se": float(values.sem()) if len(values) > 1 else None}
        )

pd.DataFrame(gene_rows).to_csv(paper.derived("Fig4A_stats_gene_pleiotropy.csv"))
pd.DataFrame(coverage_rows).to_csv(paper.derived("Fig4A_stats_gene_coverage.csv"))
pd.DataFrame(gene_rows).tail(4)

## Numbers

In [ ]:
print(paper.save_results("gene_pleiotropy", numbers))
pd.Series(numbers).to_frame("computed")